# Constraint Satisfaction Problem
## Labor- und Präsentationsplanung als CSP

### Problemstellung

An einer Hochschule soll ein Präsentationsplan für eine Projektwoche erstellt werden. Jede Projektgruppe muss drei Präsentationen (A, B, C) durchführen, die jeweils einem eindeutigen Tag, Zeitslot, Raum und einer Prüfungskommission zugeordnet werden müssen. 

Dieses Problem lässt sich formal als **Constraint Satisfaction Problem (CSP)** modellieren:

$$\text{CSP} = (X, D, C)$$

Dabei stehen die einzelnen Elemente für:
* **Variablen ($X$):** Die durchzuführenden Präsentationen der jeweiligen Projektgruppen.
* **Domänen ($D$):** Der Lösungsraum für jede Variable, bestehend aus den Kombinationen der zur Verfügung stehenden Ressourcen (3 Laborräume, 5 Tage, 4 Zeitslots, verfügbare Kommissionen).
* **Constraints ($C$):** Die Menge der Regeln, die bei der Zuweisung gelten. 

Die Constraints $C$ unterteilen sich dabei in zwei Kategorien:
1. **Harte Constraints:** Restriktionen, die zwingend erfüllt sein müssen, damit ein Plan gültig ist.
2. **Weiche Constraints:** Kriterien zur qualitativen Bewertung und Optimierung eines gültigen Plans.

### Harte Constraints

| # | Beschreibung |
|---|---|
| **H1** | Kein Raum darf zum selben Zeitpunkt doppelt belegt sein |
| **H2** | Keine Kommission darf zum selben Zeitpunkt in zwei Präsentationen sein |
| **H3** | Jede Projektgruppe darf maximal eine Präsentation pro Tag durchführen |
| **H4** | Präsentationen einer Gruppe müssen in der Reihenfolge A → B → C stattfinden |
| **H5** | Zwischen zwei aufeinanderfolgenden Präsentationen derselben Gruppe müssen mindestens 2 freie Zeitslots liegen (Indexdifferenz ≥ 3) |
| **H6** | Raumzuweisung nach Aufgabentyp: A → nur L1 oder L2, B → nur L3, C → beliebiger Raum |
| **H7** | Eine Kommission darf nur Präsentationen zu Themen bewerten, für die sie Kompetenz besitzt |
| **H8** | Eine Kommission darf nur zu Zeitpunkten eingeplant werden, an denen sie verfügbar ist |
| **H9** | Eine Kommission darf innerhalb eines Tages höchstens einmal den Raum wechseln (max. 2 verschiedene Räume pro Tag) |

### Weiche Constraints

| # | Beschreibung |
|---|---|
| **S1** | Wartezeiten der Kommissionen minimieren, Leerlaufslots zwischen aufeinanderfolgenden Präsentationen derselben Kommission am selben Tag sollen minimal sein |
| **S2** | Raumauslastung maximieren, Leerlaufslots zwischen aufeinanderfolgenden Belegungen desselben Raums am selben Tag sollen minimal sein |

### Methodischer Ansatz und Werkzeugvergleich

Um nicht nur eine *gültige*, sondern auch eine *optimale* Lösung zu finden, gliedert sich dieses Notebook in die Implementierung eines **eigenen Constraint-Validators** zur Überprüfung von Lösungen sowie die Gegenüberstellung von zwei Lösungsverfahren:

| Ansatz | Bibliothek | Ziel | Stärke |
|---|---|---|---|
| **Ansatz 1** | `python-constraint` | **Feasibility** (gültige Lösung finden) | Vorlesungsnah, transparent, ideal für den algorithmischen Vergleich |
| **Ansatz 2** | `OR-Tools CP-SAT` | **Optimierung** (beste Lösung finden) | Ermöglicht die Definition einer mathematischen Zielfunktion zur echten Minimierung der weichen Constraints |

Der zweite Ansatz wurde gezielt implementiert, da der Standard-Solver in `python-constraint` primär auf die Erfüllung harter Constraints ausgelegt ist und keine native Zielfunktion (Soft-Constraints) optimiert. OR-Tools schließt diese Lücke und erlaubt es, die Qualität der Lösungsansätze systematisch zu vergleichen und zu bewerten.

---
## 1. Konfiguration laden und analysieren

Die Konfigurationsdatei definiert alle Eingabeparameter: Gruppen, Tage, Zeitslots, Räume, Kommissionen mit Themenkompetenzen und deren Verfügbarkeiten.

| Konfiguration | Gruppen | Kommissionen | Erwartetes Ergebnis |
|---|---|---|---|
| `configB.json` | 3 | 3 | Lösbar, kleines Entwicklungsproblem |
| `configA.json` | 6 | 6 | Lösbar, Hauptkonfiguration |
| `configC.json` | 9 | 3 | Potenziell unlösbar, Stresstest |


In [1]:
from csp_utils import analyze_and_display

In [ ]:
# Ihr Dateiname - nacheinander mehrere Konfigurationen testen und analysieren.
config = analyze_and_display("configA.json")

# CSP-Konfiguration

Verwendete Datei: C:\Users\flore\Documents\GitHub\grundlagen-ki\constraints\configA.json
Lademodus: json


## Übersicht

Gruppen:
  G1, G2, G3, G4, G5, G6
Tage:
  Mon, Tue, Wed, Thu, Fri
Zeitslots:
  1: 08:00–10:00
  2: 10:30–12:30
  3: 13:00–14:30
  4: 15:00–17:00
Räume:
  L1, L2, L3
Anzahl Kommissionen:
  6
Anzahl Verfügbarkeits-Einträge:
  86


## Kompetenzen der Kommissionen

,Kommission,Themen,A,B,C
0,K1,B,,✓,
1,K2,"B, C",,✓,✓
2,K3,A,✓,,
3,K4,"A, B",✓,✓,
4,K5,C,,,✓
5,K6,"A, C",✓,,✓


## Legende

Kommission,Farbe
K1,
K2,
K3,
K4,
K5,
K6,


## Wochenplan der Verfügbarkeiten

,Mon,Tue,Wed,Thu,Fri
Zeitslot,,,,,
1 (08:00–10:00),"K2, K3, K6","K1, K2, K3, K4, K6","K1, K2, K3, K4, K5, K6","K1, K2, K3, K4, K5, K6","K1, K4, K5, K6"
2 (10:30–12:30),"K2, K3, K6","K1, K2, K3, K4, K6","K1, K2, K3, K4, K5, K6","K1, K2, K3, K4, K5, K6","K1, K4, K5, K6"
3 (13:00–14:30),"K2, K6","K1, K2, K6","K1, K2, K3, K4, K5, K6","K1, K2, K3, K4, K5, K6","K1, K6"
4 (15:00–17:00),"K2, K6","K1, K2, K6","K1, K2, K3, K4, K5, K6","K1, K2, K3, K4, K5, K6","K1, K6"


---
## 2. Ansatz 1: `python-constraint`

### 2.1 Bibliothekswahl und Verfahren

Wir verwenden `python-constraint`, da sie direkt dem Vorlesungsinhalt entspricht. Die Kernkonzepte (Variablen, Domänen, Constraint-Funktionen, Backtracking-Solver) sind transparent abgebildet. Der **Vergleich der Verfahren** findet auf Algorithmenebene statt: Wir vergleichen `BacktrackingSolver` (iterativ) und `RecursiveBacktrackingSolver` (rekursiv), beide implementieren konstruktive Constraint-Propagation mit Backtracking, unterscheiden sich aber in der Implementierung des Zustandsspeichers.

**Bekannte Einschränkung:** `python-constraint` ist ein reiner *Feasibility*-Solver ohne Zielfunktion. Die weichen Constraints lassen sich nicht direkt optimieren, dies motiviert Ansatz 2.

---

### 2.2 Variablen und Domänen

**Variablen:** Da jede Gruppe $G_i$ jedes Thema $T$ präsentieren muss, ergibt sich, dass jede Präsentation $(G_i, T)$ mit $T \in \{A, B, C\}$ eine Variable ist. Bei Config A: **18 Variablen**.

**Domänen:** Jeder Variable wird ein Tupel `(day, timeslot, room, commission)` zugewiesen. Die rohe Domänengröße beträgt theoretisch $5 \times 4 \times 3 \times |K| = 360$ Werte. Drei unäre Constraints werden als **destruktive Constraint-Propagation** bereits vor dem Solver in die Domänenkonstruktion eingebaut:

| Constraint | Beschreibung | Einbauort |
|---|---|---|
| **H6** | Raumzuweisung nach Aufgabentyp (A→L1/L2, B→L3, C→alle) | Domänenkonstruktion |
| **H7** | Kommission muss Themenkompetenz besitzen | Domänenkonstruktion |
| **H8** | Kommission muss zum Zeitpunkt verfügbar sein | Domänenkonstruktion |

Das reduziert die Domänengröße auf typischerweise **10–40 Werte** pro Variable.

---

### 2.3 Constraint-Übersicht

| Funktion | Constraints | Arität | Laufzeit-Prüfung |
|---|---|---|---|
| `no_conflict` | H1 Raumkonflikt, H2 Kommissionskonflikt | binär | $\binom{18}{2}=153$ Paare |
| `different_day` | H3 max. 1 Präsentation/Gruppe/Tag | binär | $3 \times 6=18$ Paare |
| `correct_order_with_gap` | H4 Reihenfolge A→B→C, H5 Mindestabstand | binär | $2 \times 6=12$ Paare |
| `check_h9` | H9 max. 1 Raumwechsel/Kommission/Tag | n-är | Post-Filter |

**Wichtiger Hinweis zu H3:** H5 (Mindestabstand ≥ 3 Indexeinheiten) schließt Slot 1 und Slot 4 am selben Tag nicht aus (Differenz = 3 erfüllt H5 formal). H3 ist daher ein eigenständiger, expliziter Constraint.

**H4 und H5 in einer Funktion:** Die Parameter für den Constraint `correct_order_with_gap` werden in richtiger Reihenfolge angegeben. Dadurch, dass kein Betrag der Differenz berechnet wird, ist abgesichert, dass durch den Mindestabstand ≥ 3 die korrekte Reihenfolge gewährleistet wird.

**H9 als Post-Filter:** N-ärer globaler Constraint, erst bei vollständiger Belegung prüfbar. `python-constraint` unterstützt globale Constraints nicht effizient.

**Constraint-Reihenfolge in `build_csp`:** Gruppeninterne Constraints (H3, H4, H5) werden zuerst hinzugefügt, sie schränken stark ein und lösen früh Backtracking aus. Das entspricht der Vorlesungs-Heuristik *„Variable welche in den meisten Constraints vorkommt"*.


In [ ]:
from constraint import Problem, BacktrackingSolver, RecursiveBacktrackingSolver
from collections import defaultdict
from csp_utils import DAY_ORDER
import time

TASKS = ["A", "B", "C"]

# H6: Erlaubte Räume je Aufgabentyp (statisch aus Aufgabenstellung)
ROOM_CONSTRAINTS = {
    "A": ["L1", "L2"],
    "B": ["L3"],
    "C": ["L1", "L2", "L3"],
}

def time_index(day, ts):
    """Globaler Zeitindex über die Woche: Montag Slot 1 → 1, Freitag Slot 4 → 20."""
    return DAY_ORDER[day] * 4 + int(ts)

def build_domain(task, config):
    """Vorgefilterte Domäne mit destruktiver Propagation für H6, H7, H8."""
    allowed_rooms = ROOM_CONSTRAINTS[task]                          # H6
    allowed_commissions = [                                         # H7
        k for k, topics in config["commissions"].items()
        if task in topics
    ]
    domain = set()
    for commission in allowed_commissions:
        for slot in config["availability"][commission]:             # H8
            day, ts = slot[0], int(slot[1])
            for room in allowed_rooms:
                domain.add((day, ts, room, commission))
    return list(domain)

def no_conflict(v1, v2):
    """H1 + H2: Kein Raum- oder Kommissionskonflikt zum selben Zeitpunkt."""
    if v1[0] == v2[0] and v1[1] == v2[1]:
        if v1[2] == v2[2] or v1[3] == v2[3]:
            return False
    return True

def different_day(v1, v2):
    """H3: Max. 1 Präsentation pro Gruppe pro Tag.
    Eigenständig notwendig: Slot 1 und Slot 4 selber Tag hätten Indexdifferenz 3
    und würden H5 formal erfüllen, H3 verhindert diesen Fall explizit.
    """
    return v1[0] != v2[0]

def correct_order_with_gap(v_earlier, v_later):
    """H4 + H5: Reihenfolge A→B→C und Mindestabstand von 2 freien Slots.
    Abstand ≥ 3 Indexeinheiten = mindestens 2 ungenutzte Slots zwischen Terminen.
    H4 ist implizit enthalten (positive Differenz = korrekte Reihenfolge).
    """
    return time_index(v_later[0], v_later[1]) - time_index(v_earlier[0], v_earlier[1]) >= 3

def check_h9(solution):
    """H9 (Post-Filter): Max. 1 Raumwechsel pro Kommission pro Tag.
    N-ärer Constraint, wird nach dem Solver als Filterkriterium angewendet.
    """
    commission_day_rooms = defaultdict(set)
    for val in solution.values():
        day, ts, room, commission = val
        commission_day_rooms[(commission, day)].add(room)
    return all(len(rooms) <= 2 for rooms in commission_day_rooms.values())

def build_csp(config):
    """Erstellt das CSP. Constraint-Reihenfolge: gruppeninterne Constraints zuerst
    (stark einschränkend, früh Backtracking auslösend), dann globale Paare.
    """
    problem = Problem()
    groups = config["groups"]
    all_vars = []
    for group in groups:
        for task in TASKS:
            var = f"{group}_{task}"
            problem.addVariable(var, build_domain(task, config))
            all_vars.append(var)

    for group in groups:
        vA, vB, vC = f"{group}_A", f"{group}_B", f"{group}_C"
        problem.addConstraint(different_day, [vA, vB])           # H3
        problem.addConstraint(different_day, [vA, vC])
        problem.addConstraint(different_day, [vB, vC])
        problem.addConstraint(correct_order_with_gap, [vA, vB])  # H4+H5
        problem.addConstraint(correct_order_with_gap, [vB, vC])

    for i in range(len(all_vars)):
        for j in range(i + 1, len(all_vars)):
            problem.addConstraint(no_conflict, [all_vars[i], all_vars[j]])  # H1+H2

    return problem

def solve_pc(config, solver=None, verbose=True):
    """Löst das CSP mit python-constraint und wendet H9 als Post-Filter an.
    
    Zählt, wie viele Kandidaten-Lösungen der Post-Filter verwirft, bevor
    eine H9-konforme Lösung gefunden wird. Das ist ein direktes Maß für
    den algorithmischen Preis der Generate-and-Test-Strategie.
    """
    problem = build_csp(config)
    if solver:
        problem.setSolver(solver)
    t0 = time.perf_counter()
    solution = problem.getSolution()
    elapsed = time.perf_counter() - t0
    if verbose:
        print(f"Laufzeit (getSolution): {elapsed:.3f}s")
    if solution is None:
        if verbose:
            print("Keine Lösung gefunden.")
        return None

    # Ersten Kandidat direkt prüfen
    if check_h9(solution):
        if verbose:
            print("H9 erfüllt (erster Kandidat). Post-Filter-Verwürfe: 0")
        return solution

    # Post-Filter: alle Lösungen durchsuchen, Verwürfe zählen
    if verbose:
        print("H9 verletzt, iteriere alle Lösungen...")
    t1 = time.perf_counter()
    rejected = 1  # die erste getSolution() zählt bereits
    for sol in problem.getSolutions():
        if check_h9(sol):
            elapsed2 = time.perf_counter() - t1
            if verbose:
                print(f"H9 erfüllt nach {rejected} Verwurf/Verwürfen "
                      f"({elapsed2:.3f}s für getSolutions-Iteration)")
            return sol
        rejected += 1

    if verbose:
        print(f"Keine H9-konforme Lösung unter allen {rejected} Kandidaten.")
    return None

### 2.4 Solver-Vergleich: Backtracking-Varianten

| Eigenschaft | `BacktrackingSolver` | `RecursiveBacktrackingSolver` |
|---|---|---|
| Implementierung | Iterativ (expliziter Stack) | Rekursiv (Aufrufstack) |
| Algorithmus | Identisch | Identisch |
| Speicherrisiko | Keins | Stack Overflow bei sehr tiefer Rekursion |

Beide traversieren denselben Suchbaum. Der entscheidende Laufzeitfaktor ist nicht die Solver-Variante, sondern die Qualität der Domänenreduktion: H6/H7/H8 verkleinern die Domänen von ~360 auf ~10–40 Werte, dieser Schritt hat weit mehr Einfluss auf die Laufzeit als die Wahl zwischen iterativem und rekursivem Backtracking.


In [4]:
print("=" * 55)
print("BacktrackingSolver (iterativ)")
print("=" * 55)
solution_bt = solve_pc(config, solver=BacktrackingSolver())

print()
print("=" * 55)
print("RecursiveBacktrackingSolver (rekursiv)")
print("=" * 55)
solution_rbt = solve_pc(config, solver=RecursiveBacktrackingSolver())

BacktrackingSolver (iterativ)
Laufzeit (getSolution): 0.019s
H9 erfüllt (erster Kandidat). Post-Filter-Verwürfe: 0

RecursiveBacktrackingSolver (rekursiv)
Laufzeit (getSolution): 0.015s
H9 erfüllt (erster Kandidat). Post-Filter-Verwürfe: 0


### 2.5 Ergebnisdarstellung und Beobachtung zur Nicht-Determinismus

Die Lösung wird nach Gruppe und globalem Zeitindex sortiert ausgegeben. Da `python-constraint` die *erste* gültige Lösung ohne Optimierungsziel zurückgibt, ist das Ergebnis nicht deterministisch: unterschiedliche Domänenreihenfolgen oder Solver-Varianten können zu gültig aber unterschiedlich gut bewerteten Lösungen führen. **Dies ist die zentrale Motivation für Ansatz 2.**


In [5]:
import pandas as pd
from IPython.display import display, Markdown
from csp_utils import TIMESLOT_LABELS

def display_solution(solution, label="Lösung"):
    if solution is None:
        print("Keine Lösung vorhanden.")
        return
    rows = []
    for var, (day, ts, room, commission) in solution.items():
        group, task = var.split("_")
        rows.append({"Variable": var, "Gruppe": group, "Aufgabe": task,
                     "Tag": day, "Slot": ts, "Uhrzeit": TIMESLOT_LABELS.get(ts, str(ts)),
                     "Raum": room, "Kommission": commission, "_idx": time_index(day, ts)})
    df = (pd.DataFrame(rows).sort_values(["Gruppe", "_idx"])
          .drop(columns=["_idx"]).reset_index(drop=True))
    display(Markdown(f"### {label}"))
    display(df)

display_solution(solution_bt, "Lösung (BacktrackingSolver)")

### Lösung (BacktrackingSolver)

,Variable,Gruppe,Aufgabe,Tag,Slot,Uhrzeit,Raum,Kommission
0,G1_A,G1,A,Mon,2,10:30–12:30,L1,K6
1,G1_B,G1,B,Tue,1,08:00–10:00,L3,K4
2,G1_C,G1,C,Wed,2,10:30–12:30,L2,K5
3,G2_A,G2,A,Mon,3,13:00–14:30,L2,K6
4,G2_B,G2,B,Tue,4,15:00–17:00,L3,K1
5,G2_C,G2,C,Wed,3,13:00–14:30,L2,K5
6,G3_A,G3,A,Tue,1,08:00–10:00,L1,K3
7,G3_B,G3,B,Wed,3,13:00–14:30,L3,K2
8,G3_C,G3,C,Thu,4,15:00–17:00,L3,K5
9,G4_A,G4,A,Wed,2,10:30–12:30,L1,K6


---
## 3. Constraint Validator

Der Validator prüft eine Lösung **unabhängig vom Solver** auf alle neun harten Constraints. Er dient zwei Zwecken:

1. **Korrektheitsprüfung:** Bestätigt Gültigkeit der Solver-Ausgabe, insbesondere für den extern implementierten H9.
2. **Testbarkeit:** Durch gezielte Manipulation einer Lösung können einzelne Constraints isoliert getestet werden, um die Korrektheit der Constraint-Implementierung zu verifizieren.


In [6]:
def validate(solution, config):
    """Prüft alle neun harten Constraints. Gibt True zurück wenn gültig."""
    if solution is None:
        print("Keine Lösung zum Validieren.")
        return False
    violations = []
    items = list(solution.items())

    # H1 + H2
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            n1, v1 = items[i]; n2, v2 = items[j]
            if v1[0] == v2[0] and v1[1] == v2[1]:
                if v1[2] == v2[2]:
                    violations.append(f"H1 (Raumkonflikt): {n1} & {n2} → {v1[2]} am {v1[0]} Slot {v1[1]}")
                if v1[3] == v2[3]:
                    violations.append(f"H2 (Kommissionskonflikt): {n1} & {n2} → {v1[3]} am {v1[0]} Slot {v1[1]}")
    # H3
    for group in config["groups"]:
        days = [solution[f"{group}_{t}"][0] for t in TASKS if f"{group}_{t}" in solution]
        if len(days) != len(set(days)):
            violations.append(f"H3: {group} hat zwei Präsentationen am selben Tag: {days}")
    # H4 + H5
    for group in config["groups"]:
        for t1, t2 in [("A","B"), ("B","C")]:
            va, vb = solution.get(f"{group}_{t1}"), solution.get(f"{group}_{t2}")
            if va and vb:
                diff = time_index(vb[0],vb[1]) - time_index(va[0],va[1])
                if diff < 3:
                    violations.append(f"H4/H5 ({group}): {t1}→{t2} Abstand {diff} < 3")
    # H6
    for var, (day, ts, room, commission) in solution.items():
        _, task = var.split("_")
        if room not in ROOM_CONSTRAINTS[task]:
            violations.append(f"H6: {var} → Raum {room} nicht erlaubt für Typ {task}")
    # H7
    for var, (day, ts, room, commission) in solution.items():
        _, task = var.split("_")
        if task not in config["commissions"].get(commission, []):
            violations.append(f"H7: {var} → {commission} hat keine Kompetenz für Typ {task}")
    # H8
    for var, (day, ts, room, commission) in solution.items():
        avail = [(s[0], int(s[1])) for s in config["availability"].get(commission, [])]
        if (day, ts) not in avail:
            violations.append(f"H8: {var} → {commission} nicht verfügbar am {day} Slot {ts}")
    # H9
    com_day_rooms = defaultdict(set)
    for _, (day, ts, room, commission) in solution.items():
        com_day_rooms[(commission, day)].add(room)
    for (commission, day), rooms in com_day_rooms.items():
        if len(rooms) > 2:
            violations.append(f"H9: {commission} am {day} in {len(rooms)} Räumen: {rooms}")

    print("=" * 50 + "\nCONSTRAINT VALIDATOR\n" + "=" * 50)
    if not violations:
        print("Alle harten Constraints erfüllt.")
        return True
    print(f"{len(violations)} Verletzung(en):")
    for v in violations: print(f"  * {v}")
    return False

validate(solution_bt, config)

CONSTRAINT VALIDATOR
Alle harten Constraints erfüllt.


True

---
## 4. Bewertung der weichen Constraints (Ansatz 1)

`python-constraint` kann die weichen Constraints nicht optimieren. Die Bewertungsfunktion gibt an, wie gut die gefundene Lösung in Bezug auf S1 und S2 ist, aber ohne Garantie auf Optimalität.

**Beobachtung:** Verschiedene Läufe oder Solver-Varianten können unterschiedliche Scores liefern, da keine Zielfunktion die Suche lenkt. Genau dies motiviert Ansatz 2.


In [7]:
def evaluate_soft(solution, label=""):
    """Bewertet S1 (Kommissions-Wartezeit) und S2 (Raumauslastung).
    Zählt Leerlaufslots: Slots, die zwischen zwei Belegungen desselben
    Akteurs (Kommission bzw. Raum) an einem Tag ungenutzt bleiben.
    """
    if solution is None:
        print("Keine Lösung.")
        return None, None
    com_slots, room_slots = defaultdict(list), defaultdict(list)
    for val in solution.values():
        day, ts, room, commission = val
        com_slots[(commission, day)].append(time_index(day, ts))
        room_slots[(room, day)].append(time_index(day, ts))

    def total_gap(slot_dict):
        total = 0
        for indices in slot_dict.values():
            indices.sort()
            for k in range(len(indices) - 1):
                total += indices[k+1] - indices[k] - 1
        return total

    s1 = total_gap(com_slots)
    s2 = total_gap(room_slots)
    prefix = f"[{label}] " if label else ""
    print(f"{prefix}S1 Kommissions-Wartezeit: {s1} Leerlaufslots")
    print(f"{prefix}S2 Raum-Leerlauf:         {s2} Leerlaufslots")
    print(f"{prefix}Gesamtscore:              {s1 + s2}  (niedriger = besser)")
    return s1, s2

s1_bt, s2_bt = evaluate_soft(solution_bt, "BacktrackingSolver")

[BacktrackingSolver] S1 Kommissions-Wartezeit: 3 Leerlaufslots
[BacktrackingSolver] S2 Raum-Leerlauf:         6 Leerlaufslots
[BacktrackingSolver] Gesamtscore:              9  (niedriger = besser)


---
## 5. Ansatz 2: OR-Tools CP-SAT

### 5.1 Motivation und Unterschied zu Ansatz 1

Der erste Ansatz liefert gültige, aber nicht notwendigerweise optimale Lösungen. Da `python-constraint` keine Zielfunktion unterstützt, können weiche Constraints S1 und S2 nur nachträglich bewertet, nicht aber während der Suche minimiert werden.

**OR-Tools CP-SAT** (Constraint Programming with SAT-based optimization) ist ein industrieller Solver von Google, der explizit für kombinatorische Optimierung ausgelegt ist. Er kombiniert:
- **Constraint Propagation** (wie Ansatz 1)
- **SAT-basierte Suche** (effiziente Exploration des Suchraums)
- **Branch-and-Bound** (systematische Optimierung der Zielfunktion)

Das Resultat ist eine **garantiert optimale Lösung** (OPTIMAL-Status) oder die beste in einem Zeitlimit gefundene (FEASIBLE-Status).

---

### 5.2 Modellierung: Boolesche Indikatoren

OR-Tools CP-SAT arbeitet bevorzugt mit Integer- und Boolean-Variablen. Wir nutzen dasselbe vorberechnete Domänen-Modell wie in Ansatz 1 und übersetzen es in **boolesche Indikatorvariablen**:

Für jede Präsentation $p = (G_i, T)$ und jeden Domänenwert $d_k \in D_p$:
$$b_{p,k} = 1 \iff \text{Präsentation } p \text{ wird Wert } d_k \text{ zugewiesen}$$

Mit der Bedingung:
$$\sum_{k} b_{p,k} = 1 \quad \forall p \in X \quad \text{(genau ein Wert pro Variable)}$$

Constraints werden als **Implikationsverbote** formuliert: Wenn zwei Werte $d_{p1,k1}$ und $d_{p2,k2}$ sich widersprechen, dürfen sie nicht gleichzeitig gewählt werden:
$$b_{p1,k1} + b_{p2,k2} \leq 1 \quad \text{(= AddBoolOr([b1.Not(), b2.Not()]))}$$

Diese Formulierung ist für SAT-Solver besonders effizient.

---

### 5.3 Zielfunktion: Korrekte Leerlaufslot-Zählung

Die naive Formulierung der Zielfunktion über paarweise Ko-Selektion aller Werte würde zu **Doppelzählung** führen: Bei einer Kommission, die an Slots 1, 3 und 4 eines Tages arbeitet, würden die Paare (1,4) und (3,4) beide den leeren Slot 2 als Lücke zählen.

Stattdessen verwenden wir die **„Wasted-Slot"-Formulierung**:

> Ein Slot $t_{mid}$ gilt als verschwendet, wenn:
> 1. $t_{mid}$ **nicht** belegt ist, UND
> 2. es mindestens einen belegten Slot **vor** $t_{mid}$ gibt, UND  
> 3. es mindestens einen belegten Slot **nach** $t_{mid}$ gibt.

Formal: $\text{wasted}_{t_{mid}} = \lnot \text{active}_{t_{mid}} \wedge \bigvee_{t' < t_{mid}} \text{active}_{t'} \wedge \bigvee_{t'' > t_{mid}} \text{active}_{t''}$

Diese Formulierung zählt jeden leeren Zwischenslot genau **einmal**, unabhängig von der Zahl der Belegungen. Die Zielfunktion ist die Summe aller verschwendeten Slots über alle Kommissionen und Räume.


In [8]:
from ortools.sat.python import cp_model
from itertools import combinations as iter_combinations

def build_is_active(model, b, domains, presentations, actor_fn, actor_day_slot):
    """Erstellt BoolVars is_active[(actor, day, slot)] = OR über alle
    Präsentationen, die an diesem (actor, day, slot) eingesetzt werden könnten."""
    is_active = {}
    for p in presentations:
        for i, v in enumerate(domains[p]):
            actor = actor_fn(v)
            key = (actor, v[0], v[1])
            actor_day_slot[key].append((p, i))
    for (actor, day, ts), entries in actor_day_slot.items():
        act = model.NewBoolVar(f"act_{actor}_{day}_{ts}")
        model.AddBoolOr([b[p][i] for p, i in entries]).OnlyEnforceIf(act)
        model.AddBoolAnd([b[p][i].Not() for p, i in entries]).OnlyEnforceIf(act.Not())
        is_active[(actor, day, ts)] = act
    return is_active


def add_wasted_slot_terms(model, is_active, actors, all_days, slots, label):
    """Erzeugt Zielfunktionsterme via 'Wasted-Slot'-Formulierung.

    Vermeidet Doppelzählung: Ein leerer Zwischenslot wird genau einmal gezählt,
    unabhängig davon, wie viele Belegungen ihn einrahmen. Korrekt auch bei 3+
    Belegungen pro (Akteur, Tag).
    """
    terms = []
    for actor in actors:
        for day in all_days:
            possible = [ts for ts in slots if (actor, day, ts) in is_active]
            if len(possible) < 3:
                continue
            for mid_idx in range(1, len(possible) - 1):
                ts_mid = possible[mid_idx]
                act_mid = is_active[(actor, day, ts_mid)]
                before = [is_active[(actor, day, ts)] for ts in possible[:mid_idx]]
                after  = [is_active[(actor, day, ts)] for ts in possible[mid_idx+1:]]

                has_before = model.NewBoolVar(f"hb_{label}_{actor}_{day}_{ts_mid}")
                model.AddBoolOr(before).OnlyEnforceIf(has_before)
                model.AddBoolAnd([x.Not() for x in before]).OnlyEnforceIf(has_before.Not())

                has_after = model.NewBoolVar(f"ha_{label}_{actor}_{day}_{ts_mid}")
                model.AddBoolOr(after).OnlyEnforceIf(has_after)
                model.AddBoolAnd([x.Not() for x in after]).OnlyEnforceIf(has_after.Not())

                wasted = model.NewBoolVar(f"waste_{label}_{actor}_{day}_{ts_mid}")
                model.AddBoolAnd([act_mid.Not(), has_before, has_after]).OnlyEnforceIf(wasted)
                model.AddBoolOr([act_mid, has_before.Not(), has_after.Not()]).OnlyEnforceIf(wasted.Not())
                terms.append(wasted)
    return terms


def build_ortools_csp(config):
    """Erstellt das vollständige OR-Tools CP-SAT Modell mit allen harten
    Constraints und der Wasted-Slot-Zielfunktion für S1 und S2."""
    model = cp_model.CpModel()
    groups = config["groups"]
    presentations = [(g, t) for g in groups for t in TASKS]
    domains = {p: build_domain(p[1], config) for p in presentations}
    all_days = sorted(DAY_ORDER.keys(), key=lambda d: DAY_ORDER[d])
    slots = [1, 2, 3, 4]

    # Indikatorvariablen
    b = {}
    for p in presentations:
        b[p] = [model.NewBoolVar(f"{p[0]}_{p[1]}_{i}") for i in range(len(domains[p]))]
        model.AddExactlyOne(b[p])

    # H1 + H2: Konfliktverbote als Implikationen
    for p1, p2 in iter_combinations(presentations, 2):
        for i1, v1 in enumerate(domains[p1]):
            for i2, v2 in enumerate(domains[p2]):
                if v1[0]==v2[0] and v1[1]==v2[1] and (v1[2]==v2[2] or v1[3]==v2[3]):
                    model.AddBoolOr([b[p1][i1].Not(), b[p2][i2].Not()])

    # H3: Verschiedene Tage je Gruppe
    for g in groups:
        for t1, t2 in iter_combinations(TASKS, 2):
            p1, p2 = (g, t1), (g, t2)
            for i1, v1 in enumerate(domains[p1]):
                for i2, v2 in enumerate(domains[p2]):
                    if v1[0] == v2[0]:
                        model.AddBoolOr([b[p1][i1].Not(), b[p2][i2].Not()])

    # H4 + H5: Reihenfolge und Mindestabstand
    for g in groups:
        for t1, t2 in [("A","B"), ("B","C")]:
            p1, p2 = (g, t1), (g, t2)
            for i1, v1 in enumerate(domains[p1]):
                for i2, v2 in enumerate(domains[p2]):
                    if time_index(v2[0],v2[1]) - time_index(v1[0],v1[1]) < 3:
                        model.AddBoolOr([b[p1][i1].Not(), b[p2][i2].Not()])

    # H9: Max 1 Raumwechsel pro Kommission pro Tag
    # Verbiete jede Kombination von 3 Einträgen mit 3 verschiedenen Räumen
    com_day_all = defaultdict(list)
    for p in presentations:
        for i, v in enumerate(domains[p]):
            com_day_all[(v[3], v[0])].append((p, i, v[2]))
    for (com, day), entries in com_day_all.items():
        for e1, e2, e3 in iter_combinations(entries, 3):
            if len({e1[2], e2[2], e3[2]}) == 3:
                model.Add(b[e1[0]][e1[1]] + b[e2[0]][e2[1]] + b[e3[0]][e3[1]] <= 2)

    # Zielfunktion: Wasted-Slot-Formulierung für S1 und S2
    com_day_slot  = defaultdict(list)
    room_day_slot = defaultdict(list)
    for p in presentations:
        for i, v in enumerate(domains[p]):
            com_day_slot [(v[3], v[0], v[1])].append((p, i))
            room_day_slot[(v[2], v[0], v[1])].append((p, i))

    all_coms  = sorted(config["commissions"].keys())
    all_rooms = ["L1", "L2", "L3"]

    is_act_com  = build_is_active(model, b, domains, presentations, lambda v: v[3], com_day_slot)
    is_act_room = build_is_active(model, b, domains, presentations, lambda v: v[2], room_day_slot)

    obj_terms = (add_wasted_slot_terms(model, is_act_com,  all_coms,  all_days, slots, "c") +
                 add_wasted_slot_terms(model, is_act_room, all_rooms, all_days, slots, "r"))

    if obj_terms:
        model.Minimize(sum(obj_terms))

    return model, b, domains, presentations


def solve_ortools(config, time_limit=60, verbose=True):
    """Löst das CSP mit OR-Tools CP-SAT und minimiert die weichen Constraints.
    Gibt (solution_dict, solver_object) zurück."""
    t0 = time.perf_counter()
    model, b, domains, presentations = build_ortools_csp(config)
    t_build = time.perf_counter() - t0
    if verbose:
        print(f"Modell aufgebaut in {t_build:.2f}s")

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit

    t1 = time.perf_counter()
    status = solver.Solve(model)
    t_solve = time.perf_counter() - t1

    if verbose:
        print(f"Status:             {solver.StatusName(status)}")
        print(f"Laufzeit Solver:    {t_solve:.2f}s")
        if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            print(f"Zielfunktionswert: {solver.ObjectiveValue():.0f} Leerlaufslots (optimal)")

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        if verbose: print("Keine Lösung gefunden.")
        return None, solver

    solution = {}
    for p in presentations:
        for i, val in enumerate(domains[p]):
            if solver.Value(b[p][i]):
                solution[f"{p[0]}_{p[1]}"] = val
                break

    return solution, solver

### 5.4 Ausführen und Ergebnis

In [9]:
print("=" * 55)
print("OR-Tools CP-SAT (Optimierung)")
print("=" * 55)
solution_ort, ort_solver = solve_ortools(config, time_limit=60)

display_solution(solution_ort, "Optimale Lösung (OR-Tools CP-SAT)")
validate(solution_ort, config)
s1_ort, s2_ort = evaluate_soft(solution_ort, "OR-Tools")

OR-Tools CP-SAT (Optimierung)
Modell aufgebaut in 6.02s
Status:             OPTIMAL
Laufzeit Solver:    8.23s
Zielfunktionswert: 0 Leerlaufslots (optimal)


### Optimale Lösung (OR-Tools CP-SAT)

,Variable,Gruppe,Aufgabe,Tag,Slot,Uhrzeit,Raum,Kommission
0,G1_A,G1,A,Tue,2,10:30–12:30,L1,K3
1,G1_B,G1,B,Thu,2,10:30–12:30,L3,K4
2,G1_C,G1,C,Fri,1,08:00–10:00,L3,K5
3,G2_A,G2,A,Wed,1,08:00–10:00,L2,K3
4,G2_B,G2,B,Thu,1,08:00–10:00,L3,K4
5,G2_C,G2,C,Fri,2,10:30–12:30,L2,K5
6,G3_A,G3,A,Mon,1,08:00–10:00,L1,K3
7,G3_B,G3,B,Tue,1,08:00–10:00,L3,K1
8,G3_C,G3,C,Wed,4,15:00–17:00,L1,K5
9,G4_A,G4,A,Tue,1,08:00–10:00,L2,K3


CONSTRAINT VALIDATOR
Alle harten Constraints erfüllt.
[OR-Tools] S1 Kommissions-Wartezeit: 0 Leerlaufslots
[OR-Tools] S2 Raum-Leerlauf:         0 Leerlaufslots
[OR-Tools] Gesamtscore:              0  (niedriger = besser)


---
## 6. Vergleich beider Ansätze

### 6.1 Lösungsqualität

| Kriterium | Ansatz 1: `python-constraint` | Ansatz 2: OR-Tools CP-SAT |
|---|---|---|
| **Ziel** | Erste gültige Lösung | Optimale Lösung |
| **S1/S2 Score** | Zufällig (nicht deterministisch) | Garantiert minimal |
| **Gültigkeit** | ✓ (alle harten Constraints) | ✓ (alle harten Constraints) |
| **H9** | Post-Filter | Direkt im Modell (Triplet-Verbot) |
| **Laufzeit (ConfigA)** | ~0.1–2s | ~9s |

### 6.2 Algorithmischer Vergleich

**Backtracking (Ansatz 1):** Konstruktive Propagation, belegt Variablen schrittweise, macht bei Verletzung zurück. Kein Optimierungsziel. Nicht deterministisch.

**CP-SAT (Ansatz 2):** Kombiniert Constraint Propagation, SAT-Suche und Branch-and-Bound. Die Zielfunktion lenkt die Suche aktiv in Richtung besserer Lösungen. Der Solver kann nachweisen, dass eine gefundene Lösung *optimal* ist (OPTIMAL-Status).

### 6.3 Interpretation

- Wenn beide Ansätze Score 0 liefern: die Probleminstanz ist gut genug, dass auch Backtracking ohne Optimierung eine optimale Lösung findet. Das ist ein positiver Befund über ConfigA.
- Wenn OR-Tools einen niedrigeren Score liefert: zeigt den konkreten Vorteil des Optimierungsansatzes.
- Bei Config C (unlösbar): beide Ansätze geben `None` zurück, OR-Tools liefert zusätzlich den Status INFEASIBLE, was die Unlösbarkeit formal beweist.


In [10]:
print("Vergleich der Soft-Constraint-Scores:")
print(f"  python-constraint (BacktrackingSolver): S1={s1_bt}, S2={s2_bt}, Gesamt={s1_bt+s2_bt}")
print(f"  OR-Tools CP-SAT (optimal):              S1={s1_ort}, S2={s2_ort}, Gesamt={s1_ort+s2_ort}")
if s1_bt + s2_bt > s1_ort + s2_ort:
    print("  -> OR-Tools findet eine bessere Lösung.")
elif s1_bt + s2_bt == s1_ort + s2_ort:
    print("  -> Beide Ansätze erreichen denselben Score. python-constraint findet zufällig eine optimale Lösung.")
else:
    print("  -> Unerwartetes Ergebnis: Backtracking besser als Optimierung. Bitte prüfen.")

Vergleich der Soft-Constraint-Scores:
  python-constraint (BacktrackingSolver): S1=3, S2=6, Gesamt=9
  OR-Tools CP-SAT (optimal):              S1=0, S2=0, Gesamt=0
  -> OR-Tools findet eine bessere Lösung.


---
## 7. Test aller Konfigurationen

### Erwartete Beobachtungen

- **Config B:** Beide Ansätze lösen problemlos. Laufzeiten vernachlässigbar.
- **Config A:** Beide Ansätze finden eine Lösung. OR-Tools liefert garantiert optimalen Score.
- **Config C:** Beide Ansätze finden keine Lösung (strukturell unlösbar: zu wenige Kommissionen für Typ-B-Präsentationen). OR-Tools gibt zusätzlich INFEASIBLE zurück, einen formalen Beweis der Unlösbarkeit. Zudem enthält Config C einen Datenfehler (G9 doppelt eingetragen).


In [11]:
from csp_utils import load_config

for cfg_file in ["configB.json", "configA.json", "configC.json"]:
    print(f"\n{'#'*60}\n# {cfg_file}\n{'#'*60}")
    cfg, _ = load_config(cfg_file)

    print("--- Ansatz 1: python-constraint ---")
    sol_pc = solve_pc(cfg, verbose=True)
    validate(sol_pc, cfg)
    evaluate_soft(sol_pc, "python-constraint")

    print("\n--- Ansatz 2: OR-Tools CP-SAT ---")
    sol_ort, ort_sv = solve_ortools(cfg, time_limit=30, verbose=True)
    validate(sol_ort, cfg)
    evaluate_soft(sol_ort, "OR-Tools")


############################################################
# configB.json
############################################################
--- Ansatz 1: python-constraint ---
Laufzeit (getSolution): 0.005s
H9 erfüllt (erster Kandidat). Post-Filter-Verwürfe: 0
CONSTRAINT VALIDATOR
Alle harten Constraints erfüllt.
[python-constraint] S1 Kommissions-Wartezeit: 0 Leerlaufslots
[python-constraint] S2 Raum-Leerlauf:         0 Leerlaufslots
[python-constraint] Gesamtscore:              0  (niedriger = besser)

--- Ansatz 2: OR-Tools CP-SAT ---
Modell aufgebaut in 1.06s
Status:             OPTIMAL
Laufzeit Solver:    1.27s
Zielfunktionswert: 0 Leerlaufslots (optimal)
CONSTRAINT VALIDATOR
Alle harten Constraints erfüllt.
[OR-Tools] S1 Kommissions-Wartezeit: 0 Leerlaufslots
[OR-Tools] S2 Raum-Leerlauf:         0 Leerlaufslots
[OR-Tools] Gesamtscore:              0  (niedriger = besser)

############################################################
# configA.json
###################################

---
## 8. Reflexion und Fazit

### Modellierungsentscheidungen

Das Tupel-Modell `(day, ts, room, commission)` als Variablenwert hat sich in beiden Ansätzen bewährt: Es ermöglicht eine kompakte Constraint-Formulierung und erlaubt die direkte Einbettung von H6, H7 und H8 als destruktive Propagation in die Domänenkonstruktion. Eine feingranularere Variablenstruktur wäre komplexer zu implementieren und weniger effizient.

### Vergleich der Verfahren

| Dimension | Backtracking (Ansatz 1) | CP-SAT (Ansatz 2) |
|---|---|---|
| Konzept | Konstruktive Propagation + Backtracking | Propagation + SAT + Branch-and-Bound |
| Optimierung | Nicht möglich | Zielfunktion direkt integriert |
| Determinismus | Nicht deterministisch | Deterministisch (beste Lösung) |
| Transparenz | Hoch (Vorlesungskonzepte direkt) | Mittler (black-box Internals) |
| Laufzeit (ConfigA) | ~1s | ~9s |
| Lösungsqualität | Gültig, nicht garantiert optimal | Garantiert optimal |

### Grenzen und offene Punkte

**H9 in beiden Ansätzen:** In Ansatz 1 als Post-Filter, in Ansatz 2 über Triplet-Verbote. Beide Implementierungen sind korrekt, aber H9 wäre mit einem echten globalen Constraint effizienter, z.B. über cumulative constraints in CP-SAT.

**Wasted-Slot-Zielfunktion:** Die korrekte Formulierung vermeidet Doppelzählung bei 3+ Belegungen pro (Akteur, Tag). Dies war die wichtigste algorithmische Entscheidung im OR-Tools-Modell.

**Gesamtfazit:** Der Vergleich zeigt, dass die Wahl des Verfahrens direkt von der Anforderung abhängt: Für reine Feasibility ist Backtracking schnell und transparent. Sobald Optimierungsziele ins Spiel kommen, wie in realen Planungsszenarien mit Präferenzen — ist ein Solver mit integrierter Zielfunktion klar überlegen.


### Parametrisierungen als Erkenntnisquelle

Das Testen verschiedener JSON-Dateien (Config A/B/C) ist korrekterweise ein Test unterschiedlicher **Instanzen** desselben Modells, nicht der Modellparameter selbst. Daneben lassen sich zwei echte Parametrisierungen benennen:

**Solver-Wahl in Ansatz 1 (BacktrackingSolver vs. RecursiveBacktrackingSolver)**

Der Vergleich beider Varianten ist eine klassische Parametrisierung: dasselbe Verfahren, unterschiedliche Implementierung des Zustandsspeichers (Heap vs. Aufrufstack). Das Ergebnis — identische Lösungsqualität, vernachlässigbarer Laufzeitunterschied — ist selbst eine Erkenntnis: Der eigentliche Laufzeittreiber ist die Qualität der Domänenreduktion durch H6/H7/H8, nicht die Backtracking-Variante. Erst bei pathologisch tiefer Rekursion (viele Gruppen, enge Constraints) würde `RecursiveBacktrackingSolver` durch Stack-Overflow-Risiken auffallen.

**OR-Tools Time Limit**

Das `time_limit`-Argument von `solve_ortools` ist ein direkter Modellparameter mit messbarer Auswirkung. Bei `time_limit=60` liefert CP-SAT für Config A den Status `OPTIMAL` — der Solver kann beweisen, dass keine bessere Lösung existiert. Bei `time_limit=1` bricht er deutlich früher ab: Wenn das Optimum (Score 0) in dieser Zeit noch nicht erreicht wurde, gibt CP-SAT `FEASIBLE` zurück — eine gültige, aber nicht nachweislich optimale Lösung. Das ist kein Fehler, sondern zeigt das anytime-Verhalten von Branch-and-Bound: Der Solver liefert immer die bisher beste Lösung. Für Config A, die strukturell wenig Konflikte hat, ist der Unterschied in der Praxis gering. Für eine deutlich schwieriger zu optimierende Instanz (mehr Gruppen, engere Verfügbarkeiten) würde `time_limit=1` messbar schlechtere Scores produzieren.

**H9 als Post-Filter: algorithmischer Preis**

H9 wird in Ansatz 1 nach dem Solver als Generate-and-Test angewendet. Der Zähler in `solve_pc` zeigt, wie groß der reale Preis dafür ist. Wenn der Filter **0 Verwürfe** produziert (erste Kandidatin ist bereits H9-konform), dann ist die Post-Filter-Strategie in der Praxis kostenlos — H9 ist kein engpass. Wenn hingegen **viele Verwürfe** nötig sind, bedeutet das, dass der Solver viele gültige Lösungen generiert, die erst nachträglich an H9 scheitern, was algorithmisch teurer ist als eine echte globale Constraint-Formulierung. Das Ergebnis des Zählers liefert damit direkte empirische Evidenz dafür, ob H9 als Post-Filter für die vorliegenden Instanzen praktikabel ist oder ob eine Modellreformulierung lohnenswert wäre.